# Candidate SSE Socio-Geodemographic Association

This notebook tests whether socio-geodemographic variables are associated with being a candidate SSE node. It keeps data preparation explicit in the notebook and delegates model fitting, odds-ratio tables, Wald tests, and fit statistics to `sse_detection.lib` (`sselib`).

Two complementary analysis families are fitted:

1. **Composition models** use sequence-window rows from `scotland_clustering_analysis_dataset.parquet`, joined to node-level candidate status from `node_stats`. These ask whether sequences in candidate nodes differ in age, sex, SIMD, urban/rural class, or health board composition from eligible background nodes.
2. **Node-level diversity/mixing models** use node-level entropy z-scores from `node_stats`. These ask whether candidate nodes are more or less mixed than expected for a node of the same size in the same window.

Eligible background nodes are restricted to `cluster_size >= min(candidate cluster size)`, so candidates are not compared against all singletons.

In [46]:
from pathlib import Path
import importlib
from typing import Any
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import data as ld  # noqa: E402
from sse_detection import lib as sselib  # noqa: E402E402

# Reload local package after editing source files
importlib.reload(sselib)
importlib.reload(ld)

OUTPUT_DIR = PROJECT_ROOT / "sse_detection" / "sse_outputs"
RESULT_DIR = PROJECT_ROOT / "sse_detection" / "association_outputs"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

## Model Specification

Primary models adjust for the identifiable time/variant structure:

- `C(window_idx)`
- `C(clade)` by default, via `VARIANT_ADJUSTER`

The window-level surveillance summaries (`wn_prop_sequenced`, `wn_positive_tests`) are documented below as an optional no-window-fixed-effect sensitivity specification. They are not included in the main primary models because they are constants within `window_idx`, so adding them alongside `C(window_idx)` creates rank collinearity and can prevent convergence.

Expanded composition models add each sequence's own standardised data-zone surveillance and epidemic-burden context. Expanded node-level models add node-level cluster means of the same context variables, standardised on the node analysis frame. Node-level mixing models use entropy null-model z-scores, so coefficients describe higher-than-window/size-expected mixing rather than absolute observed diversity.


In [47]:
VARIANT_ADJUSTER = "clade"  # change to "who_voc" for a coarser variant adjustment
CLUSTER_SE = "meta_cluster_id"  # robust SE clustering unit
MIXING_REFERENCE = "per 1 entropy null-model z-score"

PRIMARY_COMPOSITION_ADJUSTERS = [
    "C(window_idx)",
    f"C({VARIANT_ADJUSTER})",
]

WINDOW_SURVEILLANCE_ADJUSTERS = [
    "z_wn_prop_sequenced",
    "z_log1p_wn_positive_tests",
]

# Optional sensitivity model if you want window-level surveillance adjustment
# instead of window fixed effects. Do not combine these with C(window_idx).
SURVEILLANCE_COMPOSITION_ADJUSTERS = [
    f"C({VARIANT_ADJUSTER})",
    *WINDOW_SURVEILLANCE_ADJUSTERS,
]

EXPANDED_CONTEXT_ADJUSTERS = [
    "z_dz_cum_prop_sequenced",
    "z_dz_cum_incidence_per_capita",
    "z_dz_7d_test_positivity",
    "z_log1p_dz_cum_positive_tests",
]

EXPANDED_COMPOSITION_ADJUSTERS = (
    PRIMARY_COMPOSITION_ADJUSTERS + EXPANDED_CONTEXT_ADJUSTERS
)

PRIMARY_MIXING_ADJUSTERS = [
    "C(window_idx)",
    f"C({VARIANT_ADJUSTER})",
]

EXPANDED_MIXING_ADJUSTERS = (
    PRIMARY_MIXING_ADJUSTERS + EXPANDED_CONTEXT_ADJUSTERS
)

COMPOSITION_MODEL_SETS = {
    "primary": PRIMARY_COMPOSITION_ADJUSTERS,
    "expanded": EXPANDED_COMPOSITION_ADJUSTERS,
    # "surveillance_no_window_fe": SURVEILLANCE_COMPOSITION_ADJUSTERS,
}

MIXING_MODEL_SETS = {
    "primary": PRIMARY_MIXING_ADJUSTERS,
    "expanded": EXPANDED_MIXING_ADJUSTERS,
}

COMPOSITION_SPECS = [
    {
        "name": "sex",
        "column": "sex",
        "reference": "Male",
        "label": "Sex",
    },
    {
        "name": "age_band",
        "column": "age_band",
        "reference": "30-34",
        "fallback_references": ["35-39", "25-29"],
        "label": "Age band",
    },
    {
        "name": "simd_quintile",
        "column": "dz_simd_quintile",
        "reference": "3",
        "label": "SIMD quintile",
    },
    {
        "name": "urban_rural_class",
        "column": "dz_urban_rural_class",
        "reference": "Large Urban Areas",
        "label": "Urban/rural class",
    },
    {
        "name": "health_board",
        "column": "dz_health_board",
        "reference": "Greater Glasgow and Clyde",
        "label": "Health board",
    },
]

MIXING_FEATURES = [
    "sex_entropy_z",
    "age_entropy_z",
    "simd_entropy_z",
    "urban_rural_entropy_z",
    "health_board_entropy_z",
]

STANDARDISE_SPECS = {
    "z_wn_prop_sequenced": "wn_prop_sequenced",
    "z_log1p_wn_positive_tests": "log1p_wn_positive_tests",
    "z_dz_cum_prop_sequenced": "dz_cum_prop_sequenced",
    "z_dz_cum_incidence_per_capita": "dz_cum_incidence_per_capita",
    "z_dz_7d_test_positivity": "dz_7d_test_positivity",
    "z_log1p_dz_cum_positive_tests": "log1p_dz_cum_positive_tests",
}


def add_standardised_adjusters(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()
    if "wn_positive_tests" in out.columns:
        out["log1p_wn_positive_tests"] = np.log1p(out["wn_positive_tests"])
    if "dz_cum_positive_tests" in out.columns:
        out["log1p_dz_cum_positive_tests"] = np.log1p(out["dz_cum_positive_tests"])
    for target, source in STANDARDISE_SPECS.items():
        if source not in out.columns:
            continue
        values = out[source].astype(float)
        sd = values.std(skipna=True)
        if pd.isna(sd) or sd == 0:
            out[target] = np.nan
        else:
            out[target] = (values - values.mean(skipna=True)) / sd
    return out

## Load and Prepare Analysis Frames

`node_stats` is the source of the candidate label and the node-level mixing metrics. For composition models, sequence-window rows are loaded from the processed analysis dataset using `utils.data`, filtered to match `sse_detection.ipynb`'s retained odd windows, renumbered, and then joined to eligible node status.

In [48]:
outs = sselib.load_sse_outputs(OUTPUT_DIR)
node_stats = outs.node_stats.copy()

min_candidate_size = int(node_stats.loc[node_stats["sse_candidate"], "cluster_size"].min())
eligible_nodes = node_stats.loc[node_stats["cluster_size"].ge(min_candidate_size)].copy()

if CLUSTER_SE not in eligible_nodes.columns:
    raise KeyError(f"{CLUSTER_SE!r} is not present in node_stats.")
if VARIANT_ADJUSTER not in eligible_nodes.columns:
    raise KeyError(f"{VARIANT_ADJUSTER!r} is not present in node_stats.")

# Node-level frame for mixing models.
node_model_base = eligible_nodes.copy()
node_model_base["candidate"] = node_model_base["sse_candidate"].astype(int)
node_model_base[VARIANT_ADJUSTER] = node_model_base[VARIANT_ADJUSTER].fillna("Missing").astype(str)
node_model_base[CLUSTER_SE] = node_model_base[CLUSTER_SE].fillna(node_model_base["cluster_id"]).astype(str)
node_model_base = add_standardised_adjusters(node_model_base)

node_key = (
    eligible_nodes[["cluster_id", "sse_candidate", CLUSTER_SE, "cluster_size"]]
    .drop_duplicates("cluster_id")
)

sequence_columns = sorted({
    "window_id",
    "window_idx",
    "cluster_id",
    "sequence_id",
    "clade",
    "who_voc",
    "wn_prop_sequenced",
    "wn_positive_tests",
    "dz_cum_prop_sequenced",
    "dz_cum_incidence_per_capita",
    "dz_7d_test_positivity",
    "dz_cum_positive_tests",
    *(spec["column"] for spec in COMPOSITION_SPECS),
})

sequence_raw = ld.load_analysis_columns(sequence_columns, add_policy=False)

# Match sse_detection.ipynb: keep every other original window and renumber retained windows.
sequence_raw = sequence_raw.loc[sequence_raw["window_idx"] % 2 == 1].copy()
old_to_new = {
    old: new + 1
    for new, old in enumerate(sorted(sequence_raw["window_idx"].unique()))
}
sequence_raw["window_idx"] = sequence_raw["window_idx"].map(old_to_new)
sequence_raw["window_id"] = sequence_raw["window_idx"].apply(lambda x: f"W{x:03d}")

composition_base = sequence_raw.merge(node_key, on="cluster_id", how="inner")
composition_base["candidate"] = composition_base["sse_candidate"].astype(int)
composition_base[VARIANT_ADJUSTER] = composition_base[VARIANT_ADJUSTER].fillna("Missing").astype(str)
composition_base[CLUSTER_SE] = composition_base[CLUSTER_SE].fillna(composition_base["cluster_id"]).astype(str)
composition_base = add_standardised_adjusters(composition_base)

print(f"node_stats: {len(node_stats):,} nodes")
print(f"candidate nodes: {node_stats['sse_candidate'].sum():,}")
print(f"minimum candidate cluster size: {min_candidate_size}")
print(f"eligible nodes: {len(eligible_nodes):,}")
print(f"eligible candidates: {eligible_nodes['sse_candidate'].sum():,}")
print(f"eligible background: {(~eligible_nodes['sse_candidate']).sum():,}")
print()
print(f"composition sequence-window rows: {len(composition_base):,}")
print(f"unique sequences in composition frame: {composition_base['sequence_id'].nunique():,}")
print(f"nodes in composition frame: {composition_base['cluster_id'].nunique():,}")
print(f"candidate share in composition frame: {composition_base['candidate'].mean():.1%}")

node_stats: 99,006 nodes
candidate nodes: 10,619
minimum candidate cluster size: 3
eligible nodes: 29,706
eligible candidates: 10,619
eligible background: 19,087

composition sequence-window rows: 317,950
unique sequences in composition frame: 223,147
nodes in composition frame: 29,706
candidate share in composition frame: 51.1%


## Notebook-Side Preparation Helpers

The functions below deliberately handle data preparation in the notebook: complete-case filtering, reference-level resolution, and removal of tiny strata with no candidate/background variation. The regression module is then called only on already-prepared model frames.

In [31]:
def term_prefix(term: str) -> str:
    return term.split(",")[0]


def resolve_reference(data: pd.DataFrame, column: str, preferred, fallbacks=None):
    levels = set(data[column].dropna().astype(str))
    candidates = [preferred, *(fallbacks or [])]
    for ref in candidates:
        if ref is None:
            continue
        ref_str = str(ref)
        if ref_str in levels:
            return ref_str
        for level in levels:
            try:
                if float(level) == float(ref_str):
                    return level
            except ValueError:
                pass
    counts = data[column].dropna().astype(str).value_counts()
    if counts.empty:
        raise ValueError(f"No observed levels for {column!r}.")
    return str(counts.index[0])


def complete_case(data: pd.DataFrame, required: list[str]) -> pd.DataFrame:
    missing = [col for col in required if col not in data.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")
    return data.dropna(subset=required).copy()


def drop_nonvarying_levels(
    data: pd.DataFrame,
    columns: list[str],
    *,
    outcome: str = "candidate",
) -> tuple[pd.DataFrame, dict[str, int]]:
    d = data.copy()
    dropped: dict[str, int] = {}
    changed = True
    while changed:
        changed = False
        for col in columns:
            if col not in d.columns:
                continue
            varies = d.groupby(col, dropna=False)[outcome].transform("nunique").gt(1)
            n_drop = int((~varies).sum())
            if n_drop:
                dropped[col] = dropped.get(col, 0) + n_drop
                d = d.loc[varies].copy()
                changed = True
    return d, dropped


def add_model_metadata(table: pd.DataFrame, **metadata) -> pd.DataFrame:
    out = table.copy()
    for key, value in reversed(list(metadata.items())):
        out.insert(0, key, value)
    return out


def add_fit_metadata(fit_stats: pd.DataFrame, **metadata) -> pd.DataFrame:
    out = fit_stats.copy()
    for key, value in reversed(list(metadata.items())):
        out.insert(0, key, value)
    return out


def bh_adjust_by(table: pd.DataFrame, group_cols: list[str], p_col: str = "P>chi2") -> pd.DataFrame:
    if table.empty:
        return table.copy()
    out = table.copy()
    out["p_adj_bh"] = np.nan
    for _, idx in out.groupby(group_cols, dropna=False).groups.items():
        adjusted = sselib.bh_adjust(out.loc[idx], p_col=p_col)
        out.loc[idx, "p_adj_bh"] = adjusted["p_adj_bh"].to_numpy()
    return out


## Composition Models

Each composition predictor is first entered **one at a time**. Then all composition predictors are entered **together**. Both sets are fitted with primary adjusters and expanded adjusters. Odds ratios are sequence-level associations with candidate-node membership, with robust standard errors clustered by `meta_cluster_id`.

In [ ]:
def prepare_composition_frame(predictors: list[str], adjusters: list[str]) -> Any:
    required = (
        ["candidate", "cluster_id", "sequence_id", CLUSTER_SE]
        + predictors
        + sselib.model_variables_from_terms(adjusters)
    )
    required = list(dict.fromkeys(required))
    d = complete_case(composition_base, required)
    for col in predictors:
        d[col] = d[col].astype(str)
    strata = ["window_idx", VARIANT_ADJUSTER, *predictors]
    d, dropped = drop_nonvarying_levels(d, strata)
    return d, dropped


def fit_single_composition_models(model_set: str, adjusters: list[str]) -> Any:
    wald_tables = []
    or_tables = []
    fit_tables = []
    fitted = {}

    for spec in COMPOSITION_SPECS:
        predictor = spec["column"]
        d, dropped = prepare_composition_frame([predictor], adjusters)
        reference = resolve_reference(
            d,
            predictor,
            spec.get("reference"),
            spec.get("fallback_references"),
        )
        model_name = f"composition__{model_set}__single__{spec['name']}"
        fit = sselib.fit_exposure_model(
            d,
            outcome="candidate",
            exposure=predictor,
            adjusters=adjusters,
            model_name=model_name,
            reference=reference,
            cluster_col=CLUSTER_SE,
            categorical=True,
        )
        fitted[spec["name"]] = fit.result

        meta = {
            "domain": "composition",
            "model_set": model_set,
            "predictor_set": "single",
            "predictor": spec["name"],
            "label": spec["label"],
            "reference": reference,
            "n_model_rows": len(d),
            "n_sequences": d["sequence_id"].nunique(),
            "n_nodes": d["cluster_id"].nunique(),
            "dropped_nonvarying_rows": sum(dropped.values()),
            "dropped_nonvarying_detail": repr(dropped),
        }
        wald_tables.append(add_model_metadata(fit.wald, **meta))
        or_tables.append(add_model_metadata(fit.odds_ratios, **meta))
        fit_tables.append(add_fit_metadata(
            sselib.model_fit_stats(fit.result, model_name=model_name, formula=fit.formula),
            **meta,
        ))
        print(f"Fitted {model_name}: {len(d):,} rows", flush=True)

    return (
        fitted, pd.concat(wald_tables, ignore_index=True), 
        pd.concat(or_tables, ignore_index=True), 
        pd.concat(fit_tables, ignore_index=True)
        )


def fit_joint_composition_model(model_set: str, adjusters: list[str]) -> Any:
    predictors = [spec["column"] for spec in COMPOSITION_SPECS]
    d, dropped = prepare_composition_frame(predictors, adjusters)

    terms = []
    references = {}
    for spec in COMPOSITION_SPECS:
        reference = resolve_reference(
            d,
            spec["column"],
            spec.get("reference"),
            spec.get("fallback_references"),
        )
        references[spec["name"]] = reference
        terms.append(sselib.categorical_term(spec["column"], reference))

    formula = "candidate ~ " + " + ".join(terms + adjusters)
    model_name = f"composition__{model_set}__joint"
    result = sselib.fit_binomial_glm(d, formula, cluster_col=CLUSTER_SE)

    wald_tables = []
    for spec, term in zip(COMPOSITION_SPECS, terms):
        wald = sselib.robust_wald_for_prefix(
            result,
            term_prefix(term),
            model_name=model_name,
            term=spec["name"],
        )
        meta = {
            "domain": "composition",
            "model_set": model_set,
            "predictor_set": "joint",
            "predictor": spec["name"],
            "label": spec["label"],
            "reference": references[spec["name"]],
            "n_model_rows": len(d),
            "n_sequences": d["sequence_id"].nunique(),
            "n_nodes": d["cluster_id"].nunique(),
            "dropped_nonvarying_rows": sum(dropped.values()),
            "dropped_nonvarying_detail": repr(dropped),
        }
        wald_tables.append(add_model_metadata(wald, **meta))

    prefixes = tuple(term_prefix(term) for term in terms)
    odds = sselib.tidy_odds_ratios(result, model_name=model_name)
    odds = odds.loc[odds["term"].str.startswith(prefixes)].copy()
    odds = add_model_metadata(
        odds,
        domain="composition",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_composition",
        label="All composition predictors",
        reference=repr(references),
        n_model_rows=len(d),
        n_sequences=d["sequence_id"].nunique(),
        n_nodes=d["cluster_id"].nunique(),
        dropped_nonvarying_rows=sum(dropped.values()),
        dropped_nonvarying_detail=repr(dropped),
    )
    fit_stats = add_fit_metadata(
        sselib.model_fit_stats(result, model_name=model_name, formula=formula),
        domain="composition",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_composition",
        label="All composition predictors",
        reference=repr(references),
        n_model_rows=len(d),
        n_sequences=d["sequence_id"].nunique(),
        n_nodes=d["cluster_id"].nunique(),
        dropped_nonvarying_rows=sum(dropped.values()),
        dropped_nonvarying_detail=repr(dropped),
    )
    print(f"Fitted {model_name}: {len(d):,} rows", flush=True)
    return result, pd.concat(wald_tables, ignore_index=True), odds, fit_stats


composition_model_results = {}
composition_fits = {}


def run_composition_model_set(model_set: str, adjusters: list[str]) -> Any:
    single_fit, single_wald, single_or, single_fit_stats = fit_single_composition_models(model_set, adjusters)
    joint_fit, joint_wald, joint_or, joint_fit_stats = fit_joint_composition_model(model_set, adjusters)
    composition_fits[(model_set, "single")] = single_fit
    composition_fits[(model_set, "joint")] = joint_fit
    composition_model_results[model_set] = {
        "wald": pd.concat([single_wald, joint_wald], ignore_index=True),
        "odds": pd.concat([single_or, joint_or], ignore_index=True),
        "fit_stats": pd.concat([single_fit_stats, joint_fit_stats], ignore_index=True),
    }
    return composition_model_results[model_set]

### Primary Composition Models

These are the main single-predictor and joint composition models adjusted for window, variant, and window-level surveillance intensity.

In [49]:
composition_primary_results = run_composition_model_set(
    "primary",
    COMPOSITION_MODEL_SETS["primary"],
)

composition_primary_results["wald"][[
    "domain", "model_set", "predictor_set", "predictor", "label", "reference",
    "chi2", "df", "P>chi2", "n_model_rows", "n_sequences", "n_nodes",
    "dropped_nonvarying_detail",
]]

Fitted composition__primary__single__sex: 317,844 rows
Fitted composition__primary__single__age_band: 317,844 rows
Fitted composition__primary__single__simd_quintile: 317,844 rows
Fitted composition__primary__single__urban_rural_class: 317,844 rows
Fitted composition__primary__single__health_board: 317,844 rows
Fitted composition__primary__joint: 317,844 rows


,domain,model_set,predictor_set,predictor,label,reference,chi2,df,P>chi2,n_model_rows,n_sequences,n_nodes,dropped_nonvarying_detail
0,composition,primary,single,sex,Sex,Male,0.609390,1,0.435018,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
1,composition,primary,single,age_band,Age band,30-34,37.876209,15,0.000941,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
2,composition,primary,single,simd_quintile,SIMD quintile,3,2.897069,4,0.575196,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
3,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,25.263924,5,0.000124,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
4,composition,primary,single,health_board,Health board,Greater Glasgow and Clyde,31.485614,13,0.002862,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
5,composition,primary,joint,sex,Sex,Male,0.485737,1,0.485835,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
6,composition,primary,joint,age_band,Age band,30-34,39.656260,15,0.000511,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
7,composition,primary,joint,simd_quintile,SIMD quintile,3,2.301152,4,0.680559,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
8,composition,primary,joint,urban_rural_class,Urban/rural class,Large Urban Areas,17.879436,5,0.003101,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
9,composition,primary,joint,health_board,Health board,Greater Glasgow and Clyde,23.173727,13,0.039638,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"


### Expanded Composition Models

These repeat the composition models after adding sequence data-zone surveillance and epidemic-burden context. Treat these as a robustness/sensitivity specification.

In [50]:
composition_expanded_results = run_composition_model_set(
    "expanded",
    COMPOSITION_MODEL_SETS["expanded"],
)

composition_expanded_results["wald"][[
    "domain", "model_set", "predictor_set", "predictor", "label", "reference",
    "chi2", "df", "P>chi2", "n_model_rows", "n_sequences", "n_nodes",
    "dropped_nonvarying_detail",
]]

Fitted composition__expanded__single__sex: 317,818 rows
Fitted composition__expanded__single__age_band: 317,818 rows
Fitted composition__expanded__single__simd_quintile: 317,818 rows
Fitted composition__expanded__single__urban_rural_class: 317,818 rows
Fitted composition__expanded__single__health_board: 317,818 rows
Fitted composition__expanded__joint: 317,818 rows


,domain,model_set,predictor_set,predictor,label,reference,chi2,df,P>chi2,n_model_rows,n_sequences,n_nodes,dropped_nonvarying_detail
0,composition,expanded,single,sex,Sex,Male,0.350185,1,5.540082e-01,317818,223065,29688,"{'window_idx': 34, 'clade': 72}"
1,composition,expanded,single,age_band,Age band,30-34,43.763393,15,1.197442e-04,317818,223065,29688,"{'window_idx': 34, 'clade': 72}"
2,composition,expanded,single,simd_quintile,SIMD quintile,3,14.822992,4,5.082780e-03,317818,223065,29688,"{'window_idx': 34, 'clade': 72}"
3,composition,expanded,single,urban_rural_class,Urban/rural class,Large Urban Areas,110.898751,5,2.645730e-22,317818,223065,29688,"{'window_idx': 34, 'clade': 72}"
4,composition,expanded,single,health_board,Health board,Greater Glasgow and Clyde,82.395484,13,3.900355e-12,317818,223065,29688,"{'window_idx': 34, 'clade': 72}"
5,composition,expanded,joint,sex,Sex,Male,0.361788,1,5.475150e-01,317818,223065,29688,"{'window_idx': 34, 'clade': 72}"
6,composition,expanded,joint,age_band,Age band,30-34,44.282306,15,9.931305e-05,317818,223065,29688,"{'window_idx': 34, 'clade': 72}"
7,composition,expanded,joint,simd_quintile,SIMD quintile,3,12.740329,4,1.261682e-02,317818,223065,29688,"{'window_idx': 34, 'clade': 72}"
8,composition,expanded,joint,urban_rural_class,Urban/rural class,Large Urban Areas,54.908888,5,1.362932e-10,317818,223065,29688,"{'window_idx': 34, 'clade': 72}"
9,composition,expanded,joint,health_board,Health board,Greater Glasgow and Clyde,42.592085,13,5.238527e-05,317818,223065,29688,"{'window_idx': 34, 'clade': 72}"


### Composition Summary Tables

The omnibus Wald table is the main screening table. McFadden pseudo-R2 is included for relative comparison within this model family.

In [35]:
if not composition_model_results:
    raise RuntimeError("Run at least one composition model-set cell before summarising.")

composition_wald = pd.concat(
    [result["wald"] for result in composition_model_results.values()],
    ignore_index=True,
)
composition_wald = bh_adjust_by(composition_wald, ["domain", "model_set", "predictor_set"])
composition_or = pd.concat(
    [result["odds"] for result in composition_model_results.values()],
    ignore_index=True,
)
composition_fit_stats = pd.concat(
    [result["fit_stats"] for result in composition_model_results.values()],
    ignore_index=True,
)

display(composition_wald[[
    "domain", "model_set", "predictor_set", "predictor", "label", "reference",
    "chi2", "df", "P>chi2", "p_adj_bh", "n_model_rows", "n_sequences", "n_nodes",
    "dropped_nonvarying_detail",
]])

display(composition_fit_stats[[
    "domain", "model_set", "predictor_set", "predictor", "r2_mcfadden", "converged",
    "aic", "bic_llf", "log_likelihood", "ll_null", "n_model_rows", "n_sequences", "n_nodes",
]])

,domain,model_set,predictor_set,predictor,label,reference,chi2,df,P>chi2,p_adj_bh,n_model_rows,n_sequences,n_nodes,dropped_nonvarying_detail
0,composition,primary,single,sex,Sex,Male,0.609390,1,4.350176e-01,5.437720e-01,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
1,composition,primary,single,age_band,Age band,30-34,37.876209,15,9.410808e-04,2.352702e-03,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
2,composition,primary,single,simd_quintile,SIMD quintile,3,2.897069,4,5.751958e-01,5.751958e-01,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
3,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,25.263924,5,1.238986e-04,6.194930e-04,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
4,composition,primary,single,health_board,Health board,Greater Glasgow and Clyde,31.485614,13,2.861541e-03,4.769235e-03,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
5,composition,primary,joint,sex,Sex,Male,0.485737,1,4.858349e-01,6.072937e-01,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
6,composition,primary,joint,age_band,Age band,30-34,39.656260,15,5.109458e-04,2.554729e-03,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
7,composition,primary,joint,simd_quintile,SIMD quintile,3,2.301152,4,6.805594e-01,6.805594e-01,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
8,composition,primary,joint,urban_rural_class,Urban/rural class,Large Urban Areas,17.879436,5,3.101362e-03,7.753405e-03,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"
9,composition,primary,joint,health_board,Health board,Greater Glasgow and Clyde,23.173727,13,3.963810e-02,6.606350e-02,317844,223083,29689,"{'window_idx': 34, 'clade': 72}"


,domain,model_set,predictor_set,predictor,r2_mcfadden,converged,aic,bic_llf,log_likelihood,ll_null,n_model_rows,n_sequences,n_nodes
0,composition,primary,single,sex,0.069119,True,410186.364154,411103.925328,-205007.182077,-220229.293450,317844,223083,29689
1,composition,primary,single,age_band,0.069210,True,410174.472206,411241.403804,-204987.236103,-220229.293450,317844,223083,29689
2,composition,primary,single,simd_quintile,0.069129,True,410188.293373,411137.862495,-205005.146687,-220229.293450,317844,223083,29689
3,composition,primary,single,urban_rural_class,0.069284,True,410121.639230,411081.877668,-204970.819615,-220229.293450,317844,223083,29689
4,composition,primary,single,health_board,0.069384,True,410093.650703,411139.243669,-204948.825351,-220229.293450,317844,223083,29689
5,composition,primary,joint,all_composition,0.069577,True,410058.660115,411370.985980,-204906.330058,-220229.293450,317844,223083,29689
6,composition,expanded,single,sex,0.071247,True,409224.208630,410184.439706,-204522.104315,-220211.447982,317818,223065,29688
7,composition,expanded,single,age_band,0.071364,True,409200.468667,410310.069021,-204496.234334,-220211.447982,317818,223065,29688
8,composition,expanded,single,simd_quintile,0.071284,True,409213.864551,410206.103329,-204513.932276,-220211.447982,317818,223065,29688
9,composition,expanded,single,urban_rural_class,0.071967,True,408914.861511,409917.769523,-204363.430755,-220211.447982,317818,223065,29688


### Composition Odds Ratios

The table below contains coefficient-level odds ratios for the composition models. For primary interpretation, start with the omnibus Wald table above; use this table to identify which levels drive an omnibus association.

In [54]:
display(composition_or[[
    "domain", "model_set", "predictor_set", "predictor", "label", "reference",
    "term", "estimate", "std_error", "p_value", "odds_ratio", "or_low", "or_high",
]].sort_values(["model_set", "predictor_set", "predictor", "p_value"]))

,domain,model_set,predictor_set,predictor,label,reference,term,estimate,std_error,p_value,odds_ratio,or_low,or_high
137,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_urban_rural_class, Treatment(reference='L...",0.207290,0.031947,8.665989e-11,1.230339,1.155664,1.309840
136,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_urban_rural_class, Treatment(reference='L...",0.088201,0.016086,4.181748e-08,1.092207,1.058309,1.127192
138,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_urban_rural_class, Treatment(reference='L...",0.171918,0.033752,3.513453e-07,1.187581,1.111562,1.268799
135,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_urban_rural_class, Treatment(reference='L...",0.088500,0.020781,2.056672e-05,1.092535,1.048929,1.137953
145,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_health_board, Treatment(reference='Greate...",0.171317,0.043815,9.228064e-05,1.186866,1.089198,1.293293
...,...,...,...,...,...,...,...,...,...,...,...,...,...
22,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,"C(dz_urban_rural_class, Treatment(reference='L...",0.061525,0.014855,3.447842e-05,1.063457,1.032940,1.094875
23,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,"C(dz_urban_rural_class, Treatment(reference='L...",0.095438,0.030433,1.712783e-03,1.100141,1.036438,1.167759
24,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,"C(dz_urban_rural_class, Treatment(reference='L...",0.110420,0.036236,2.309820e-03,1.116747,1.040184,1.198945
21,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,"C(dz_urban_rural_class, Treatment(reference='L...",0.046967,0.019989,1.879019e-02,1.048087,1.007820,1.089963


## Node-Level Diversity / Mixing Models

Node-level mixing models use entropy null-model z-scores. Coefficients are reported per one-unit increase in the z-score, so they compare nodes that are more or less mixed than expected for their size and window. Primary models adjust for window and variant; expanded models additionally adjust for standardised node-level cluster means of data-zone surveillance and burden variables.


Do **not** add candidate-defining quantities such as `cluster_size`, `core_amplification_score`, `out_strength`, or `onward_dissemination_score` to the main models, because that would condition on the machinery used to define the outcome.

In [ ]:
def prepare_mixing_frame(predictors: list[str], adjusters: list[str]) -> Any:
    required = (
        ["candidate", "cluster_id", CLUSTER_SE]
        + predictors
        + sselib.model_variables_from_terms(adjusters)
    )
    required = list(dict.fromkeys(required))
    d = complete_case(node_model_base, required)
    strata = ["window_idx", VARIANT_ADJUSTER]
    d, dropped = drop_nonvarying_levels(d, strata)
    return d, dropped


def fit_single_mixing_models(model_set: str, adjusters: list[str]) -> Any:
    wald_tables = []
    or_tables = []
    fit_tables = []
    fitted = {}

    for feature in MIXING_FEATURES:
        if feature not in node_model_base.columns:
            print(f"Skipping {feature}: not found", flush=True)
            continue
        d, dropped = prepare_mixing_frame([feature], adjusters)
        model_name = f"mixing__{model_set}__single__{feature}"
        fit = sselib.fit_exposure_model(
            d,
            outcome="candidate",
            exposure=feature,
            adjusters=adjusters,
            model_name=model_name,
            cluster_col=CLUSTER_SE,
            categorical=False,
        )
        fitted[feature] = fit.result
        meta = {
            "domain": "node_mixing",
            "model_set": model_set,
            "predictor_set": "single",
            "predictor": feature,
            "label": feature.replace("_", " "),
            "reference": MIXING_REFERENCE,
            "n_model_rows": len(d),
            "n_nodes": d["cluster_id"].nunique(),
            "dropped_nonvarying_rows": sum(dropped.values()),
            "dropped_nonvarying_detail": repr(dropped),
        }
        wald_tables.append(add_model_metadata(fit.wald, **meta))
        or_tables.append(add_model_metadata(
            fit.odds_ratios.loc[fit.odds_ratios["term"].eq(feature)].copy(),
            **meta,
        ))
        fit_tables.append(add_fit_metadata(
            sselib.model_fit_stats(fit.result, model_name=model_name, formula=fit.formula),
            **meta,
        ))
        print(f"Fitted {model_name}: {len(d):,} nodes", flush=True)
    return (
        fitted, pd.concat(wald_tables, ignore_index=True), 
        pd.concat(or_tables, ignore_index=True), 
        pd.concat(fit_tables, ignore_index=True)
        )


def fit_joint_mixing_model(model_set: str, adjusters: list[str]) -> Any:
    features = [feature for feature in MIXING_FEATURES if feature in node_model_base.columns]
    d, dropped = prepare_mixing_frame(features, adjusters)
    formula = "candidate ~ " + " + ".join(features + adjusters)
    model_name = f"mixing__{model_set}__joint"
    result = sselib.fit_binomial_glm(d, formula, cluster_col=CLUSTER_SE)

    wald = sselib.tidy_single_parameter_wald(result, features, model_name=model_name)
    wald = add_model_metadata(
        wald,
        domain="node_mixing",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_mixing",
        label="All mixing predictors",
        reference=MIXING_REFERENCE,
        n_model_rows=len(d),
        n_nodes=d["cluster_id"].nunique(),
        dropped_nonvarying_rows=sum(dropped.values()),
        dropped_nonvarying_detail=repr(dropped),
    )
    odds = sselib.tidy_odds_ratios(result, model_name=model_name)
    odds = odds.loc[odds["term"].isin(features)].copy()
    odds = add_model_metadata(
        odds,
        domain="node_mixing",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_mixing",
        label="All mixing predictors",
        reference=MIXING_REFERENCE,
        n_model_rows=len(d),
        n_nodes=d["cluster_id"].nunique(),
        dropped_nonvarying_rows=sum(dropped.values()),
        dropped_nonvarying_detail=repr(dropped),
    )
    fit_stats = add_fit_metadata(
        sselib.model_fit_stats(result, model_name=model_name, formula=formula),
        domain="node_mixing",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_mixing",
        label="All mixing predictors",
        reference=MIXING_REFERENCE,
        n_model_rows=len(d),
        n_nodes=d["cluster_id"].nunique(),
        dropped_nonvarying_rows=sum(dropped.values()),
        dropped_nonvarying_detail=repr(dropped),
    )
    print(f"Fitted {model_name}: {len(d):,} nodes", flush=True)
    return result, wald, odds, fit_stats

In [43]:
mixing_model_results = {}
mixing_fits = {}


def run_mixing_model_set(model_set: str, adjusters: list[str]) -> Any:
    single_fit, single_wald, single_or, single_fit_stats = fit_single_mixing_models(model_set, adjusters)
    joint_fit, joint_wald, joint_or, joint_fit_stats = fit_joint_mixing_model(model_set, adjusters)
    mixing_fits[(model_set, "single")] = single_fit
    mixing_fits[(model_set, "joint")] = joint_fit
    mixing_model_results[model_set] = {
        "wald": pd.concat([single_wald, joint_wald], ignore_index=True),
        "odds": pd.concat([single_or, joint_or], ignore_index=True),
        "fit_stats": pd.concat([single_fit_stats, joint_fit_stats], ignore_index=True),
    }
    return mixing_model_results[model_set]


### Primary Mixing Models

These node-level models test each entropy alone and then all entropies together, adjusted for window and variant.

In [51]:
mixing_primary_results = run_mixing_model_set(
    "primary",
    MIXING_MODEL_SETS["primary"],
)

mixing_primary_results["wald"][[
    "domain", "model_set", "predictor_set", "predictor", "term",
    "chi2", "df", "P>chi2", "n_model_rows", "n_nodes",
    "dropped_nonvarying_detail",
]]

Fitted mixing__primary__single__sex_entropy_obs: 29,689 nodes
Fitted mixing__primary__single__age_entropy_obs: 29,689 nodes
Fitted mixing__primary__single__simd_entropy_obs: 29,689 nodes
Fitted mixing__primary__single__urban_rural_entropy_obs: 29,689 nodes
Fitted mixing__primary__single__health_board_entropy_obs: 29,689 nodes
Fitted mixing__primary__joint: 29,689 nodes


,domain,model_set,predictor_set,predictor,term,chi2,df,P>chi2,n_model_rows,n_nodes,dropped_nonvarying_detail
0,node_mixing,primary,single,sex_entropy_obs,sex_entropy_obs,531.007977,1,1.704740e-117,29689,29689,"{'window_idx': 3, 'clade': 14}"
1,node_mixing,primary,single,age_entropy_obs,age_entropy_obs,2507.540315,1,0.000000e+00,29689,29689,"{'window_idx': 3, 'clade': 14}"
2,node_mixing,primary,single,simd_entropy_obs,simd_entropy_obs,2150.022822,1,0.000000e+00,29689,29689,"{'window_idx': 3, 'clade': 14}"
3,node_mixing,primary,single,urban_rural_entropy_obs,urban_rural_entropy_obs,1601.057086,1,0.000000e+00,29689,29689,"{'window_idx': 3, 'clade': 14}"
4,node_mixing,primary,single,health_board_entropy_obs,health_board_entropy_obs,1299.013504,1,1.851831e-284,29689,29689,"{'window_idx': 3, 'clade': 14}"
5,node_mixing,primary,joint,all_mixing,sex_entropy_obs,8.712157,1,3.160949e-03,29689,29689,"{'window_idx': 3, 'clade': 14}"
6,node_mixing,primary,joint,all_mixing,age_entropy_obs,707.764126,1,6.128541e-156,29689,29689,"{'window_idx': 3, 'clade': 14}"
7,node_mixing,primary,joint,all_mixing,simd_entropy_obs,183.780008,1,7.246651e-42,29689,29689,"{'window_idx': 3, 'clade': 14}"
8,node_mixing,primary,joint,all_mixing,urban_rural_entropy_obs,43.648681,1,3.929418e-11,29689,29689,"{'window_idx': 3, 'clade': 14}"
9,node_mixing,primary,joint,all_mixing,health_board_entropy_obs,21.321763,1,3.882981e-06,29689,29689,"{'window_idx': 3, 'clade': 14}"


### Expanded Mixing Models

These add node-level cluster means of the data-zone context variables, matching the expanded composition sensitivity model.

In [52]:
mixing_expanded_results = run_mixing_model_set(
    "expanded",
    MIXING_MODEL_SETS["expanded"],
)

mixing_expanded_results["wald"][[
    "domain", "model_set", "predictor_set", "predictor", "term",
    "chi2", "df", "P>chi2", "n_model_rows", "n_nodes",
    "dropped_nonvarying_detail",
]]

Fitted mixing__expanded__single__sex_entropy_obs: 29,688 nodes
Fitted mixing__expanded__single__age_entropy_obs: 29,688 nodes
Fitted mixing__expanded__single__simd_entropy_obs: 29,688 nodes
Fitted mixing__expanded__single__urban_rural_entropy_obs: 29,688 nodes
Fitted mixing__expanded__single__health_board_entropy_obs: 29,688 nodes
Fitted mixing__expanded__joint: 29,688 nodes


,domain,model_set,predictor_set,predictor,term,chi2,df,P>chi2,n_model_rows,n_nodes,dropped_nonvarying_detail
0,node_mixing,expanded,single,sex_entropy_obs,sex_entropy_obs,526.035264,1,2.058281e-116,29688,29688,"{'window_idx': 3, 'clade': 14}"
1,node_mixing,expanded,single,age_entropy_obs,age_entropy_obs,2478.800526,1,0.000000e+00,29688,29688,"{'window_idx': 3, 'clade': 14}"
2,node_mixing,expanded,single,simd_entropy_obs,simd_entropy_obs,2116.156300,1,0.000000e+00,29688,29688,"{'window_idx': 3, 'clade': 14}"
3,node_mixing,expanded,single,urban_rural_entropy_obs,urban_rural_entropy_obs,1647.895454,1,0.000000e+00,29688,29688,"{'window_idx': 3, 'clade': 14}"
4,node_mixing,expanded,single,health_board_entropy_obs,health_board_entropy_obs,1231.665824,1,8.007673e-270,29688,29688,"{'window_idx': 3, 'clade': 14}"
5,node_mixing,expanded,joint,all_mixing,sex_entropy_obs,9.169172,1,2.461259e-03,29688,29688,"{'window_idx': 3, 'clade': 14}"
6,node_mixing,expanded,joint,all_mixing,age_entropy_obs,679.539718,1,8.414356e-150,29688,29688,"{'window_idx': 3, 'clade': 14}"
7,node_mixing,expanded,joint,all_mixing,simd_entropy_obs,177.581080,1,1.635265e-40,29688,29688,"{'window_idx': 3, 'clade': 14}"
8,node_mixing,expanded,joint,all_mixing,urban_rural_entropy_obs,73.632444,1,9.410514e-18,29688,29688,"{'window_idx': 3, 'clade': 14}"
9,node_mixing,expanded,joint,all_mixing,health_board_entropy_obs,37.759734,1,8.001644e-10,29688,29688,"{'window_idx': 3, 'clade': 14}"


### Mixing Summary Tables

The Wald table gives the per-feature evidence. The odds-ratio table below gives the direction and magnitude per 0.1 increase in normalised entropy.

In [40]:
if not mixing_model_results:
    raise RuntimeError("Run at least one mixing model-set cell before summarising.")

mixing_wald = pd.concat(
    [result["wald"] for result in mixing_model_results.values()],
    ignore_index=True,
)
mixing_wald = bh_adjust_by(mixing_wald, ["domain", "model_set", "predictor_set"])
mixing_or = pd.concat(
    [result["odds"] for result in mixing_model_results.values()],
    ignore_index=True,
)
mixing_fit_stats = pd.concat(
    [result["fit_stats"] for result in mixing_model_results.values()],
    ignore_index=True,
)

display(mixing_wald[[
    "domain", "model_set", "predictor_set", "predictor", "term",
    "chi2", "df", "P>chi2", "p_adj_bh", "n_model_rows", "n_nodes",
    "dropped_nonvarying_detail",
]])

display(mixing_fit_stats[[
    "domain", "model_set", "predictor_set", "predictor", "r2_mcfadden", "converged",
    "aic", "bic_llf", "log_likelihood", "ll_null", "n_model_rows", "n_nodes",
]])

,domain,model_set,predictor_set,predictor,term,chi2,df,P>chi2,p_adj_bh,n_model_rows,n_nodes,dropped_nonvarying_detail
0,node_mixing,primary,single,sex_entropy_obs,sex_entropy_obs,531.007977,1,1.704740e-117,1.704740e-117,29689,29689,"{'window_idx': 3, 'clade': 14}"
1,node_mixing,primary,single,age_entropy_obs,age_entropy_obs,2507.540315,1,0.000000e+00,0.000000e+00,29689,29689,"{'window_idx': 3, 'clade': 14}"
2,node_mixing,primary,single,simd_entropy_obs,simd_entropy_obs,2150.022822,1,0.000000e+00,0.000000e+00,29689,29689,"{'window_idx': 3, 'clade': 14}"
3,node_mixing,primary,single,urban_rural_entropy_obs,urban_rural_entropy_obs,1601.057086,1,0.000000e+00,0.000000e+00,29689,29689,"{'window_idx': 3, 'clade': 14}"
4,node_mixing,primary,single,health_board_entropy_obs,health_board_entropy_obs,1299.013504,1,1.851831e-284,2.314788e-284,29689,29689,"{'window_idx': 3, 'clade': 14}"
5,node_mixing,primary,joint,all_mixing,sex_entropy_obs,8.712157,1,3.160949e-03,3.160949e-03,29689,29689,"{'window_idx': 3, 'clade': 14}"
6,node_mixing,primary,joint,all_mixing,age_entropy_obs,707.764126,1,6.128541e-156,3.064270e-155,29689,29689,"{'window_idx': 3, 'clade': 14}"
7,node_mixing,primary,joint,all_mixing,simd_entropy_obs,183.780008,1,7.246651e-42,1.811663e-41,29689,29689,"{'window_idx': 3, 'clade': 14}"
8,node_mixing,primary,joint,all_mixing,urban_rural_entropy_obs,43.648681,1,3.929418e-11,6.549030e-11,29689,29689,"{'window_idx': 3, 'clade': 14}"
9,node_mixing,primary,joint,all_mixing,health_board_entropy_obs,21.321763,1,3.882981e-06,4.853726e-06,29689,29689,"{'window_idx': 3, 'clade': 14}"


,domain,model_set,predictor_set,predictor,r2_mcfadden,converged,aic,bic_llf,log_likelihood,ll_null,n_model_rows,n_nodes
0,node_mixing,primary,single,sex_entropy_obs,0.056151,True,36713.105399,37426.779142,-18270.552700,-19357.503828,29689,29689
1,node_mixing,primary,single,age_entropy_obs,0.108942,True,34669.300444,35382.974186,-17248.650222,-19357.503828,29689,29689
2,node_mixing,primary,single,simd_entropy_obs,0.093898,True,35251.734099,35965.407842,-17539.867050,-19357.503828,29689,29689
3,node_mixing,primary,single,urban_rural_entropy_obs,0.075021,True,35982.555355,36696.229097,-17905.277677,-19357.503828,29689,29689
4,node_mixing,primary,single,health_board_entropy_obs,0.067781,True,36262.873048,36976.546790,-18045.436524,-19357.503828,29689,29689
5,node_mixing,primary,joint,all_mixing,0.115032,True,34441.534646,35188.402516,-17130.767323,-19357.503828,29689,29689
6,node_mixing,expanded,single,sex_entropy_obs,0.059801,True,36577.886313,37324.751151,-18198.943157,-19356.475383,29688,29688
7,node_mixing,expanded,single,age_entropy_obs,0.111680,True,34569.490619,35316.355458,-17194.745310,-19356.475383,29688,29688
8,node_mixing,expanded,single,simd_entropy_obs,0.097157,True,35131.728238,35878.593076,-17475.864119,-19356.475383,29688,29688
9,node_mixing,expanded,single,urban_rural_entropy_obs,0.080508,True,35776.243171,36523.108009,-17798.121586,-19356.475383,29688,29688


### Mixing Odds Ratios

For entropy z-score models, odds ratios are per one-unit increase in the entropy z-score. OR < 1 means candidate nodes are less likely at higher-than-null mixing; OR > 1 means candidate nodes are more likely at higher-than-null mixing.

In [41]:
display(mixing_or[[
    "domain", "model_set", "predictor_set", "predictor", "term",
    "estimate", "std_error", "p_value", "odds_ratio", "or_low", "or_high",
]].sort_values(["model_set", "predictor_set", "predictor", "p_value"]))

,domain,model_set,predictor_set,predictor,term,estimate,std_error,p_value,odds_ratio,or_low,or_high
16,node_mixing,expanded,joint,all_mixing,age_entropy_obs,2.335626,0.089597,8.414356e-150,10.335928,8.671295,12.320122
17,node_mixing,expanded,joint,all_mixing,simd_entropy_obs,0.825330,0.061934,1.635265e-40,2.282635,2.021706,2.577239
18,node_mixing,expanded,joint,all_mixing,urban_rural_entropy_obs,0.539900,0.062919,9.410514e-18,1.715836,1.516769,1.941029
19,node_mixing,expanded,joint,all_mixing,health_board_entropy_obs,-0.457350,0.074428,8.001644e-10,0.632959,0.547044,0.732366
15,node_mixing,expanded,joint,all_mixing,sex_entropy_obs,0.118491,0.039131,2.461259e-03,1.125797,1.042682,1.215538
11,node_mixing,expanded,single,age_entropy_obs,age_entropy_obs,3.253922,0.065356,0.000000e+00,25.891699,22.778719,29.430104
14,node_mixing,expanded,single,health_board_entropy_obs,health_board_entropy_obs,1.876145,0.053459,8.007673e-270,6.528287,5.878885,7.249424
10,node_mixing,expanded,single,sex_entropy_obs,sex_entropy_obs,0.820159,0.035759,2.058281e-116,2.270861,2.117152,2.435730
12,node_mixing,expanded,single,simd_entropy_obs,simd_entropy_obs,2.101039,0.045673,0.000000e+00,8.174659,7.474681,8.940188
13,node_mixing,expanded,single,urban_rural_entropy_obs,urban_rural_entropy_obs,1.874823,0.046184,0.000000e+00,6.519667,5.955430,7.137361


## Interpretation Guide

Use the tables in this order:

1. **Single-predictor primary models**: main adjusted association for each socio-geodemographic variable, controlling for time and variant context.
2. **Single-predictor expanded models**: sensitivity to local data-zone surveillance and epidemic-burden adjustment.
3. **Joint primary models**: whether each predictor retains signal when the socio-geodemographic predictors are mutually adjusted.
4. **Joint expanded models**: the most conditional specification; useful as a robustness check rather than the simplest effect summary.
5. **McFadden pseudo-R2**: compare models within the same outcome/family and analysis frame. Small values are normal for logistic models; the main use here is relative comparison between primary vs expanded and single vs joint models.

Composition models describe **who/where the sequences in candidate nodes come from**. Node-level mixing models describe **whether candidate nodes are unusually internally diverse or concentrated** for their size and window.

In [53]:
summary_tables = {
    "composition_wald.csv": composition_wald,
    "composition_odds_ratios.csv": composition_or,
    "composition_fit_stats.csv": composition_fit_stats,
    "mixing_wald.csv": mixing_wald,
    "mixing_odds_ratios.csv": mixing_or,
    "mixing_fit_stats.csv": mixing_fit_stats,
}


def clean_export_table(table: pd.DataFrame) -> pd.DataFrame:
    out = table.copy()
    out.columns = [str(col).strip() for col in out.columns]
    for col in out.select_dtypes(include=["object", "string"]).columns:
        present = out[col].notna()
        out.loc[present, col] = out.loc[present, col].astype(str).str.strip()
    return out


for filename, table in summary_tables.items():
    path = RESULT_DIR / filename
    clean_export_table(table).to_csv(path, index=False)
    print(f"saved {filename}: {len(table):,} rows")

saved composition_wald.csv: 20 rows
saved composition_odds_ratios.csv: 152 rows
saved composition_fit_stats.csv: 12 rows
saved mixing_wald.csv: 20 rows
saved mixing_odds_ratios.csv: 20 rows
saved mixing_fit_stats.csv: 12 rows
